======================================================================================================================

$$\text{This script Reproduce paired equilibrium/flow sensitivity grids in the original and capacity-scaled regimes}$$
$$\text{Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$\text{31/07/2026}$$

======================================================================================================================

In [3]:
from concurrent.futures import ProcessPoolExecutor,as_completed
from pathlib import Path
from csv import DictWriter
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
import numpy as np,time
OUT=Path(__file__).resolve().parent
RGRID=np.linspace(.08,.50,15); LAMGRID=np.linspace(.15,.85,21)
NGRID=(2.,5.,9.); XGRID=(.05,.20,.40); GAMMAS=(1.,50.,150.,500.)
KAPPA=.3; S0=.9; T=30.; EPS=.4

def simulate(g,r,lam,n,x0,feedback):
 d=g*r;c=1+x0
 def rhs(t,y):
  s,x=y;f=s/(KAPPA+s); k=max(c-x,0.)**n
  return [d*(k if feedback else 1.)*(1-s)-g*f*x,(f-lam)*x]
 def closure(t,y):return c-y[1]
 closure.direction=-1;closure.terminal=True
 def tolerance(t,y):return max(c-y[1],0.)**n-EPS
 tolerance.direction=0;tolerance.terminal=False
 sol=solve_ivp(rhs,(0,T),(S0,x0),events=[closure,tolerance],dense_output=True,
               method='LSODA',rtol=2e-8,atol=2e-10)
 if not sol.success:return {'status':'failure'}
 if len(sol.t_events[0]):return {'status':'closure'}
 ts=np.unique(np.r_[0,sol.t_events[1],T]);
 duration=float(sum(b-a for a,b in zip(ts[:-1],ts[1:]) if max(c-sol.sol((a+b)*.5)[1],0.)**n<=EPS))
 idx=int(np.argmax(sol.y[1]));lo=sol.t[max(idx-1,0)];hi=sol.t[min(idx+1,len(sol.t)-1)]
 xmax=sol.y[1,idx]
 if hi>lo:
  rs=minimize_scalar(lambda z:-float(sol.sol(z)[1]),bounds=(lo,hi),method='bounded',options={'xatol':1e-8});xmax=max(xmax,float(sol.sol(rs.x)[1]))
 t_eval=np.linspace(0,T,151);series=np.maximum(c-sol.sol(t_eval)[1],0.)**n
 return {'status':'full','kmin':float(max(c-xmax,0.)**n),'duration':duration,'series':series,'final':float(series[-1])}

def batch(g,x0,n):
 rows=[];t0=time.time()
 for lam in LAMGRID:
  for r in RGRID:
   persistent=lam*(1+KAPPA)<1
   H=r*(1-lam*(1+KAPPA))/(lam*(1-lam)) if persistent else np.nan
   f=simulate(g,r,lam,n,x0,True);c=simulate(g,r,lam,n,x0,False)
   row=dict(gamma=g,x0=x0,n=n,lambda_val=lam,r=r,delta=r*g,
            R0=1/(lam*(1+KAPPA)),persistent=int(persistent),H=H,
            H_below_x0=int(persistent and H<x0),status_FB=f['status'],status_CF=c['status'])
   if f['status']==c['status']=='full':
    row.update(status='full',kmin_FB=f['kmin'],kmin_CF=c['kmin'],
               duration_FB=f['duration'],duration_CF=c['duration'],
               delta_kmin=f['kmin']-c['kmin'],delta_duration=c['duration']-f['duration'],
               delta_auc=float(np.trapezoid(f['series']-c['series'],dx=T/150)),
               delta_final=f['final']-c['final'])
   else:row['status']='censored'
   rows.append(row)
 print('completed',g,x0,n,'in',round(time.time()-t0,2),'seconds',flush=True)
 return rows

def main():
 tasks=[(g,x0,n) for g in GAMMAS for x0 in XGRID for n in NGRID]; rows=[]
 with ProcessPoolExecutor(max_workers=4) as pool:
  futures={pool.submit(batch,*v):v for v in tasks}
  for f in as_completed(futures):rows.extend(f.result())
 cols=sorted({a for row in rows for a in row})
 with (OUT/'physical_scale_sweep_full.csv').open('w',newline='') as stream:
  wr=DictWriter(stream,fieldnames=cols);wr.writeheader();wr.writerows(rows)
 summaries=[]
 for g in GAMMAS:
  for x0 in XGRID:
   subset=[row for row in rows if row['gamma']==g and row['x0']==x0]
   full=[row for row in subset if row['status']=='full'];persistent=[row for row in subset if row['persistent']]
   z=dict(gamma=g,x0=x0,pairs=len(subset),persistent=len(persistent),
          H_below_x0=sum(a['H_below_x0'] for a in subset),
          H_below_share=sum(a['H_below_x0'] for a in subset)/len(persistent),
          complete=len(full),CF_closures=sum(a['status_CF']=='closure' for a in subset),
          FB_closures=sum(a['status_FB']=='closure' for a in subset),
          positive_delta_kmin=sum(a['delta_kmin']>1e-6 for a in full),
          positive_delta_duration=sum(a['delta_duration']>1e-6 for a in full),
          positive_delta_auc=sum(a['delta_auc']>1e-6 for a in full),
          negative_delta_auc=sum(a['delta_auc']< -1e-6 for a in full),
          mean_delta_auc=float(np.mean([a['delta_auc'] for a in full])))
   summaries.append(z)
 with (OUT/'physical_scale_sweep_summary.csv').open('w',newline='') as stream:
  wr=DictWriter(stream,fieldnames=list(summaries[0]));wr.writeheader();wr.writerows(summaries)
 for z in summaries:print('SUMMARY',z,flush=True)

if __name__=='__main__':main()


NameError: name '__file__' is not defined